This notebook will check whether the increase in profit shiftings stem from the inflation

In [100]:
import pandas as pd
import tjn_tools

In [101]:
# Retrieve profits shiftings outwards 2021 (2017 data) and 2022 (2018 data)
pso_2021 = pd.read_excel('../data/intermediate/profits_shifting_outward_2021.xlsx')
pso_2022 = pd.read_excel('../data/intermediate/profits_shifting_outward_2022.xlsx')
pso_2021

,iso3,Country,Shifted profits outward (USD million)
0,DZA,Algeria,1655
1,AGO,Angola,601
2,BEN,Benin,211
3,BWA,Botswana,23
4,BFA,Burkina Faso,0
...,...,...,...
187,PLW,Palau,107
188,PNG,Papua New Guinea,213
189,WSM,Samoa,249
190,SLB,Solomon Islands,1


In [102]:
# Retrieve inflation data
inflation = pd.read_csv('../data/raw/20230308_world_bank_unilateral_panel.csv')
# Keep only relevant columns
inflation = inflation.loc[:,['iso3','year','inflation']]
# Keep only 2018 inflation data
inflation = inflation.loc[inflation['year']==2018]
# Estimate missing inflation values using the global average
inflation.fillna(value={"inflation": inflation['inflation'].mean()}, inplace=True)
# Clean dataset
inflation.rename(columns={'inflation':'inflation_2018'}, inplace=True)
inflation.drop(columns='year', inplace=True)

inflation

,iso3,inflation_2018
20,ABW,3.626041
45,AFG,0.626149
70,AGO,19.630594
95,AIA,4.328283
120,ALA,4.328283
...,...,...
6245,XXK,4.328283
6270,YEM,4.328283
6295,ZAF,4.517165
6320,ZMB,7.494572


In [103]:
# Calculate adjusted 2017 data
pso_2022_adjusted = pd.merge(pso_2022, inflation, on='iso3', how='left')
pso_2022_adjusted.insert(3, 'Adjusted Shifted profits outward (USD million)', pso_2022_adjusted['Shifted profits outward (USD million)']* (1-((pso_2022_adjusted['inflation_2018']/100))))
pso_2022_adjusted


,iso3,Country,Shifted profits outward (USD million),Adjusted Shifted profits outward (USD million),inflation_2018
0,ABW,Aruba,18,17.347313,3.626041
1,AFG,Afghanistan,0,0.000000,0.626149
2,AGO,Angola,503,404.258110,19.630594
3,AIA,Anguilla,0,0.000000,4.328283
4,ALB,Albania,321,314.489929,2.028060
...,...,...,...,...,...
208,XKX,Kosovo,14,13.852468,1.053798
209,YEM,Yemen,2,1.913434,4.328283
210,ZAF,South Africa,10855,10364.661715,4.517165
211,ZMB,Zambia,2759,2552.224761,7.494572


In [104]:
# Compare profit shifting between 2018 data and adjusted 2017 data
profits_shifting_2018 = pso_2022_adjusted['Shifted profits outward (USD million)'].sum()
profits_shifting_2018_adjusted = int(pso_2022_adjusted['Adjusted Shifted profits outward (USD million)'].sum())

print('Profits shifted in 2018 (in millions): ', profits_shifting_2018)
print('Profits shifted in 2018 with inflation adjustment (in millions): ', profits_shifting_2018_adjusted)
print('Increase in shifted in 2018 explained by inflation (in millions): ',profits_shifting_2018 - profits_shifting_2018_adjusted)

Profits shifted in 2018 (in millions):  1576993
Profits shifted in 2018 with inflation adjustment (in millions):  1535844
Increase in shifted in 2018 explained by inflation (in millions):  41149
